In [5]:
!pip install huggingface_hub
!pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 22.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 60.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]


In [6]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
)
from peft import LoraConfig
from trl import SFTTrainer

In [10]:
from huggingface_hub import login

login("hf_your_token")

In [13]:
from huggingface_hub import snapshot_download

repo_path = snapshot_download(
    repo_id="Daleth-hb/cuda-hip-gpu-dataset",
    repo_type="model"
)

print(repo_path)

Fetching 4 files: 100%|████████████████████████████████████████████████████████████| 4/4 [00:01<00:00,  3.90it/s]

/root/.cache/huggingface/hub/models--Daleth-hb--cuda-hip-gpu-dataset/snapshots/15151035e8b5fd53c129fe3fb3f4e38015369e30


In [15]:
from datasets import load_dataset

dataset_translation = load_dataset(
    "json",
    data_files=f"{repo_path}/cuda_to_hip_dataset.jsonl",
    split="train"
)

Generating train split: 1133 examples [00:00, 16248.65 examples/s]


In [16]:
repo_path_raft = snapshot_download(
    repo_id="Daleth-hb/cuda-hip-gpu-RAFT-dataset",
    repo_type="model"
)

dataset_reasoning = load_dataset(
    "json",
    data_files=f"{repo_path_raft}/cuda_hip_raft_dataset.jsonl",
    split="train"
)

Fetching 3 files: 100%|████████████████████████████████████████████████████████████| 3/3 [00:00<00:00,  4.35it/s]
Generating train split: 1133 examples [00:00, 15520.05 examples/s]


In [17]:
from datasets import concatenate_datasets

dataset = concatenate_datasets([
    dataset_translation,
    dataset_reasoning
]).shuffle(seed=42)

In [21]:
def format_example(example):
    return {
        "text": f"""### Task
Convert the following CUDA code to ROCm HIP.

### CUDA
{example['input']}

### HIP
{example['output']}"""
    }

dataset = dataset.map(format_example)

Map: 100%|██████████████████████████████████████████████████████████| 2266/2266 [00:00<00:00, 7650.08 examples/s]


In [22]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2-7B-Instruct",
    trust_remote_code=True
)

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=4096
    )

dataset = dataset.map(tokenize)

Map: 100%|████████████████████████████████████████████████████████████| 2266/2266 [00:23<00:00, 97.92 examples/s]


In [23]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-Coder-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█████████████████████████████████████████████████████████| 339/339 [00:04<00:00, 71.16it/s]


In [24]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 10,092,544 || all params: 7,625,709,056 || trainable%: 0.1323


In [26]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./qwen_cuda2hip_lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=2,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    bf16=True,
    optim="adamw_torch",
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

trainer.train()

Step,Training Loss
10,0.468885
20,0.423563
30,0.402251
40,0.404261
50,0.411252
60,0.382165
70,0.458923
80,0.398112
90,0.368978
100,0.397224


TrainOutput(global_step=568, training_loss=0.3561468611300831, metrics={'train_runtime': 1527.4513, 'train_samples_per_second': 2.967, 'train_steps_per_second': 0.372, 'total_flos': 5.918243203876147e+17, 'train_loss': 0.3561468611300831, 'epoch': 2.0})

In [27]:
model.save_pretrained("./qwen_cuda2hip_lora")
tokenizer.save_pretrained("./qwen_cuda2hip_lora")

('./qwen_cuda2hip_lora/tokenizer_config.json',
 './qwen_cuda2hip_lora/chat_template.jinja',
 './qwen_cuda2hip_lora/tokenizer.json')

In [28]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="./qwen_cuda2hip_lora",
    tokenizer="./qwen_cuda2hip_lora",
    device_map="auto"
)

prompt = """
### Task
Convert the following CUDA code to ROCm HIP.

### CUDA
__global__ void add(int *a, int *b, int *c){
    int i = threadIdx.x;
    c[i] = a[i] + b[i];
}

### HIP
"""

result = pipe(prompt, max_new_tokens=200)

print(result[0]["generated_text"])

Loading weights: 100%|██████████████████████████████████████████████████████| 224/224 [00:00<00:00, 10222.22it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_th


### Task
Convert the following CUDA code to ROCm HIP.

### CUDA
__global__ void add(int *a, int *b, int *c){
    int i = threadIdx.x;
    c[i] = a[i] + b[i];
}

### HIP
```cpp
#include "hip/hip_runtime.h"
__global__ void add(int *a, int *b, int *c){
    int i = threadIdx.x;
    c[i] = a[i] + b[i];
}
```

### Migration Reasoning:
Migrating from CUDA to ROCm (HIP) involves several steps, including updating the API calls and ensuring compatibility with the ROCm platform. Below is a detailed explanation of each step:

### Step 1: Include the Correct Header File
In CUDA, you include `<cuda_runtime.h>`. For ROCm, you need to include `<hip/hip_runtime.h>` instead. This header file provides the necessary functions and types for interacting with the ROCm runtime environment.

**CUDA Code:**
```cpp
#include <cuda_runtime.h>
```

**HIP Code:**
```cpp
#include "hip/hip_runtime.h"
```

### Step 2: Update Kernel Launch Syntax
The syntax for launching kernels remains


In [29]:
model.push_to_hub("Daleth-hb/qwen-cuda2hip-lora")
tokenizer.push_to_hub("Daleth-hb/qwen-cuda2hip-lora")

Processing Files (0 / 0): |                                                         |  0.00B /  0.00B            
Processing Files (0 / 1):   3%|█▍                                                   | 1.12MB / 40.4MB,   ???B/s  
Processing Files (0 / 1):  99%|████████████████████████████████████████████████████▋| 40.2MB / 40.4MB,   ???B/s  
Processing Files (1 / 1): 100%|█████████████████████████████████████████████████████| 40.4MB / 40.4MB, 3.79MB/s  
Processing Files (1 / 1): 100%|█████████████████████████████████████████████████████| 40.4MB / 40.4MB, 3.79MB/s  
New Data Upload: 100%|██████████████████████████████████████████████████████████████| 40.4MB / 40.4MB, 3.79MB/s  
Processing Files (0 / 0): |                                                         |  0.00B /  0.00B            
Processing Files (0 / 1): 100%|████████████████████████████████████████████████████▊| 11.4MB / 11.4MB,   ???B/s  
Processing Files (1 / 1): 100%|█████████████████████████████████████████████████████| 11

CommitInfo(commit_url='https://huggingface.co/Daleth-hb/qwen-cuda2hip-lora/commit/104c4d63715dfb68875bb755d565206df55c30bd', commit_message='Upload tokenizer', commit_description='', oid='104c4d63715dfb68875bb755d565206df55c30bd', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Daleth-hb/qwen-cuda2hip-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='Daleth-hb/qwen-cuda2hip-lora'), pr_revision=None, pr_num=None)